# Split COCO JSON Annotations
Reads the original `coco.json` and creates a separate `coco.json` inside each of the `train/`, `val/`, and `test/` split folders, containing only the images and annotations that belong to that split.

In [1]:
import os
import json

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DATA_PATH  = './data'
COCO_JSON_PATH  = os.path.join(BASE_DATA_PATH, 'Dataset', 'coco.json')
SPLIT_BASE_DIR  = os.path.join(BASE_DATA_PATH, 'split_dataset')
SPLITS          = ['train', 'val', 'test']

print(f"COCO JSON source : {COCO_JSON_PATH}")
print(f"Split base dir   : {SPLIT_BASE_DIR}")

COCO JSON source : ./data/Dataset/coco.json
Split base dir   : ./data/split_dataset


In [2]:
# ── Load master COCO JSON ──────────────────────────────────────────────────
if not os.path.exists(COCO_JSON_PATH):
    raise FileNotFoundError(f"COCO JSON not found: {COCO_JSON_PATH}")

with open(COCO_JSON_PATH, 'r') as f:
    coco = json.load(f)

print(f"Loaded coco.json")
print(f"  Images      : {len(coco['images'])}")
print(f"  Annotations : {len(coco['annotations'])}")
print(f"  Categories  : {len(coco['categories'])}")

Loaded coco.json
  Images      : 10417
  Annotations : 75336
  Categories  : 10


In [3]:
# ── Build lookup: filename (no ext) → COCO image entry ────────────────────
filename_to_coco_image = {}
for img_entry in coco['images']:
    base = os.path.splitext(img_entry['file_name'])[0]
    filename_to_coco_image[base] = img_entry

# Build lookup: image_id → list of annotations
image_id_to_annotations = {}
for ann in coco['annotations']:
    image_id_to_annotations.setdefault(ann['image_id'], []).append(ann)

print(f"Lookup tables built for {len(filename_to_coco_image)} images")

Lookup tables built for 10417 images


In [4]:
# ── Create a coco.json for each split ─────────────────────────────────────
for split in SPLITS:
    img_dir    = os.path.join(SPLIT_BASE_DIR, split, 'images')
    output_dir = os.path.join(SPLIT_BASE_DIR, split)
    output_path = os.path.join(output_dir, 'coco.json')

    if not os.path.exists(img_dir):
        print(f"[SKIPPED] {split}: images folder not found at {img_dir}")
        continue

    # collect filenames in this split (without extension)
    split_filenames = {
        os.path.splitext(f)[0]
        for f in os.listdir(img_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    }

    split_images      = []
    split_annotations = []
    missing           = []

    for base_name in split_filenames:
        coco_img = filename_to_coco_image.get(base_name)
        if coco_img is None:
            missing.append(base_name)
            continue

        split_images.append(coco_img)
        split_annotations.extend(image_id_to_annotations.get(coco_img['id'], []))

    split_coco = {
        'info'       : coco.get('info', {}),
        'licenses'   : coco.get('licenses', []),
        'categories' : coco['categories'],
        'images'     : split_images,
        'annotations': split_annotations,
    }

    with open(output_path, 'w') as f:
        json.dump(split_coco, f)

    print(f"[SUCCESS] {split:5s} → {output_path}")
    print(f"          images={len(split_images):4d}  annotations={len(split_annotations):5d}", end="")
    if missing:
        print(f"  [WARNING] {len(missing)} image(s) not found in coco.json", end="")
    print()

print("\nDone — each split folder now contains its own coco.json")

[SUCCESS] train → ./data/split_dataset/train/coco.json
          images=7291  annotations=52556
[SUCCESS] val   → ./data/split_dataset/val/coco.json
          images=1563  annotations=11347
[SUCCESS] test  → ./data/split_dataset/test/coco.json
          images=1563  annotations=11433

Done — each split folder now contains its own coco.json
